In [3]:
import numpy as np
import random
import time
from IPython.display import clear_output

# --- 1. AYARLAR VE MATEMATİKSEL ALTYAPI ---
MAP_6x6 = [
    "+-----------+",
    "|R: | : : :G|",
    "| : | : : : |",
    "| : : : : : |",
    "| | : | : : |",
    "|Y| : |S: : |",
    "| : : : :B: |",
    "+-----------+",
]

def encode_state(t_row, t_col, pass_idx, dest_idx):
    i = t_row; i *= 6; i += t_col; i *= 6; i += pass_idx; i *= 5; i += dest_idx
    return i

def decode_state(state):
    out = []
    out.append(state % 5); state //= 5
    out.append(state % 6); state //= 6
    out.append(state % 6); state //= 6
    out.append(state)
    return list(reversed(out))

# Değişkenleri ilklendir
q_table = np.zeros([1080, 6])
all_rewards = []
all_epochs = []

# --- 2. EĞİTİM SÜRECİ (VERİ TOPLAMA DAHİL) ---
alpha, gamma, epsilon = 0.1, 0.9, 0.2

print("Eğitim başladı... Yaklaşık 1 dakika sürebilir.")

for i in range(1, 150001):
    t_row, t_col = random.randint(0,5), random.randint(0,5)
    p_idx, d_idx = random.randint(0,5), random.randint(0,4)
    state = encode_state(t_row, t_col, p_idx, d_idx)
    
    reward_sum, epochs = 0, 0
    terminated = False
    
    for _ in range(100):
        # Karar aşaması
        action = random.randint(0, 5) if random.uniform(0, 1) < epsilon else np.argmax(q_table[state])
        
        tr, tc, pi, di = decode_state(state)
        new_tr, new_tc = tr, tc
        
        # Hareket kuralları
        if action == 0 and tr < 5: new_tr += 1
        elif action == 1 and tr > 0: new_tr -= 1
        elif action == 2 and tc < 5: new_tc += 1
        elif action == 3 and tc > 0: new_tc -= 1
        
        next_state = encode_state(new_tr, new_tc, pi, di)
        
        # Ödüllendirme
        step_reward = -1
        if action >= 4: step_reward = -10
        if new_tr == tr and new_tc == tc and action < 4: step_reward = -5
        
        reward_sum += step_reward
        
        # Q-Öğrenme Güncellemesi (Bellman Denklemi)
        q_table[state, action] = (1 - alpha) * q_table[state, action] + \
                                 alpha * (step_reward + gamma * np.max(q_table[next_state]))
        
        state = next_state
        epochs += 1
        if random.random() < 0.05: break 
    
    # İstatistik kaydet (Hata almamak için her bölümü değil, örneklemi alıyoruz)
    if i % 200 == 0:
        all_rewards.append(reward_sum)
        all_epochs.append(epochs)

print("Eğitim başarıyla tamamlandı!\n")

# --- 3. ANALİZ VE ASCII GRAFİK (ÇÖKMEYEN VERSİYON) ---
def draw_ascii_chart(data, title, symbol="█"):
    print(f"{'='*20} {title} {'='*20}")
    # Veriyi seyrelt
    chunk = len(data) // 50
    points = [np.mean(data[i:i+chunk]) for i in range(0, len(data), chunk)]
    min_v, max_v = min(points), max(points)
    rng = (max_v - min_v) if max_v != min_v else 1
    
    for r in range(10, 0, -1):
        threshold = min_v + (r / 10) * rng
        line = f"{int(threshold):>6} | "
        for p in points:
            line += symbol if p >= threshold else " "
        print(line)
    print("       " + "-" * len(points) + "\n")

# Grafikleri Çiz
draw_ascii_chart(all_rewards, "ÖĞRENME BAŞARISI (ÖDÜL ARTIŞI)")
draw_ascii_chart(all_epochs, "VERİMLİLİK ANALİZİ (ADIM SAYISI AZALIŞI)", "▒")

# --- 4. ÖĞRENME RAPORU ---
print(f"Başlangıç Ort. Ödül: {np.mean(all_rewards[:10]):.2f}")
print(f"Bitiş Ort. Ödül:     {np.mean(all_rewards[-10:]):.2f}")

Eğitim başladı... Yaklaşık 1 dakika sürebilir.
Eğitim başarıyla tamamlandı!

==================== ÖĞRENME BAŞARISI (ÖDÜL ARTIŞI) ====================
   -15 |                                                   
   -19 |          █            █   █                       
   -23 |          █ █        █ █   █     █ █               
   -27 |   █      █ ██  █   ██ █ █ █   █ █ █ ███           
   -31 |   █   █  █ ██ ███  ████ █ █  ██ █ ██████  █     █ 
   -35 |  ██   ███████████  ████ ████ ██ ████████  █ █   █ 
   -40 |  ██ █ ███████████  ████ ████ ██ ████████  █ ██  ██
   -44 |  ████ █████████████████ ████ ██ █████████ █ ██████
   -48 |  ████ █████████████████ ████ ████████████ ████████
   -52 |  █████████████████████████████████████████████████
       --------------------------------------------------

==================== VERİMLİLİK ANALİZİ (ADIM SAYISI AZALIŞI) ====================
    32 |                        ▒                          
    29 | ▒    ▒                 ▒               

In [ ]:
import time
from IPython.display import clear_output

# Durak isimleri ve koordinatları (Bilgi amaçlı)
# 0:R, 1:G, 2:Y, 3:B, 4:S
durak_isimleri = ["KIRMIZI (R)", "YEŞİL (G)", "SARI (Y)", "MAVİ (B)", "GÜMÜŞ (S)"]

try:
    while True: # Hücreyi durdurana kadar döngü devam eder (GIF etkisi)
        # Her döngüde yeni bir görev: Rastgele başlangıç ve rastgele hedef
        t_row, t_col = random.randint(0, 5), random.randint(0, 5)
        p_idx = random.randint(0, 5) # Yolcu konumu
        d_idx = random.randint(0, 4) # Hedef durak
        
        current_state = encode_state(t_row, t_col, p_idx, d_idx)
        hedef_durak = durak_isimleri[d_idx]
        
        # Bir bölüm (episode) simülasyonu
        for t in range(25): # Ajan akıllıysa 25 adımdan önce bitirir
            action = np.argmax(q_table[current_state])
            
            # --- GÖRSELLEŞTİRME ---
            tr, tc, pi, di = decode_state(current_state)
            clear_output(wait=True)
            
            display_map = [list(line) for line in MAP_6x6]
            # Taksiyi sarı blok olarak yerleştir
            display_map[tr + 1][tc * 2 + 1] = "\033[43m \033[0m"
            
            print(f"🚀 SİMÜLASYON ÇALIŞIYOR | Hedef: {hedef_durak}")
            print("------------------------------------------")
            for line in display_map:
                print("".join(line))
            
            act_names = ["GÜNEY", "KUZEY", "DOĞU", "BATI", "AL", "BIRAK"]
            print(f"------------------------------------------")
            print(f"Adım: {t} | Yapılan Karar: {act_names[action]}")
            print("Durdurmak için yukarıdaki 'Kare (Stop)' butonuna basın.")
            
            # --- DURUM GÜNCELLEME ---
            new_tr, new_tc = tr, tc
            if action == 0 and tr < 5: new_tr += 1
            elif action == 1 and tr > 0: new_tr -= 1
            elif action == 2 and tc < 5: new_tc += 1
            elif action == 3 and tc > 0: new_tc -= 1
            
            current_state = encode_state(new_tr, new_tc, pi, di)
            
            time.sleep(0.3) # Kare hızı (FPS)
            
            # Eğer ajan 'Bırak' (5) aksiyonu yaptıysa bölümü bitir ve yeni göreve geç
            if action == 5:
                time.sleep(0.8) # Başarı anını görmemiz için duraksama
                break
                
except KeyboardInterrupt:
    print("\nSimülasyon kullanıcı tarafından durduruldu.")

🚀 SİMÜLASYON ÇALIŞIYOR | Hedef: YEŞİL (G)
------------------------------------------
+-----------+
|R: | : : :G|
| : | : : : |
| : : : : : |
| | : | : : |
|Y| : |S: : |
| : : : :B: |
+-----------+
------------------------------------------
Adım: 6 | Yapılan Karar: GÜNEY
Durdurmak için yukarıdaki 'Kare (Stop)' butonuna basın.
